# Single-Stock DK Trend Demo

This notebook reads local DuckDB daily bars, plots the latest DK trend, and compares the three supported signal modes.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.backtest.single_stock import run_single_stock_backtest
from src.data_fetcher.db_manager import DuckDBManager
from src.indicators import DKTrendParams, TrendMode, compute_dktrend
from src.settings import load_config

SYMBOL = "600930"
START = "2023-01-01"

In [ ]:
cfg = load_config(ROOT / "config.yaml")
with DuckDBManager(config_path=ROOT / "config.yaml") as db:
    daily = db.read_daily_frame(symbols=[SYMBOL])

daily = daily[daily["trade_date"] >= pd.Timestamp(START)].sort_values("trade_date")
daily.tail()

In [ ]:
params = DKTrendParams.from_mapping(cfg.get("trend_signal", {}))
trend = compute_dktrend(daily, params)
latest = trend[trend["dk_color"].isin(["red", "green"])].iloc[-1]
latest[["trade_date", "close", "dk_color", "dk_signal", "dk_run_len"]]

In [ ]:
plot_df = trend.tail(120).copy()
colors = plot_df["dk_color"].map({"red": "#d62728", "green": "#2ca02c"}).fillna("#999999")

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(plot_df["trade_date"], plot_df["close"], color="#222222", linewidth=1.5, label="Close")
ax.scatter(plot_df["trade_date"], plot_df["close"], c=colors, s=18, label="DK color")
ax.set_title(f"{SYMBOL} DK Trend ({params.mode.value})")
ax.grid(alpha=0.25)
ax.legend()
fig.autofmt_xdate()
plt.show()

In [ ]:
rows = []
for mode in TrendMode:
    mode_params = DKTrendParams.from_mapping({**cfg.get("trend_signal", {}), "mode": mode.value})
    result = run_single_stock_backtest(daily, params=mode_params, stock_name=SYMBOL)
    rows.append(
        {
            "mode": mode.value,
            "total_return": result.total_return,
            "buy_hold_return": result.buy_hold_return,
            "excess_return": result.excess_return,
            "sharpe": result.sharpe,
            "max_drawdown": result.max_drawdown,
            "n_trades": result.n_trades,
            "win_rate": result.win_rate,
        }
    )

pd.DataFrame(rows)